# 🏺 QwenCleo-ASR — FastAPI Server

Egyptian Arabic & code-switching speech recognition, built on Qwen3-ASR.

[Model](https://huggingface.co/mohammedaly22/QwenCleo-ASR) · [GitHub](https://github.com/MohammedAly22/qwencleo-asr) · [PyPI](https://pypi.org/project/qwencleo-asr/)

> **Runtime → Change runtime type → GPU** before running. Then run the cells top to bottom.

Run the QwenCleo FastAPI server inside Colab and call it over HTTP.

## 1. Install (server extras) + get the repo code

In [ ]:
!pip install -q 'qwencleo-asr[server]'
# the server app lives in the GitHub repo
!git clone -q https://github.com/MohammedAly22/qwencleo-asr.git
%cd qwencleo-asr

## 2. Launch the server in the background

In [ ]:
import subprocess, os, time
env = dict(os.environ, QWENCLEO_MODEL='mohammedaly22/QwenCleo-ASR')
proc = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    env=env)
print('starting server (first run downloads the model)...')
time.sleep(60)   # give it time to load the model

## 3. Health check

In [ ]:
import requests
print(requests.get('http://localhost:8000/health').json())

## 4. Transcribe a file via the API

In [ ]:
# Grab a sample Egyptian/code-switching clip to transcribe
import urllib.request, os
URL = 'https://huggingface.co/mohammedaly22/QwenCleo-ASR/resolve/main/assets/sample.wav'
FALLBACK = 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav'
path = 'sample.wav'
try:
    urllib.request.urlretrieve(URL, path)
except Exception:
    urllib.request.urlretrieve(FALLBACK, path)
print('saved', path, os.path.getsize(path), 'bytes')

In [ ]:
import requests
with open('sample.wav', 'rb') as f:
    r = requests.post('http://localhost:8000/v1/transcribe',
                      files={'file': f},
                      data={'language': 'Arabic'})
print(r.json())

## 5. Stop the server

In [ ]:
proc.terminate()